In [1]:
import warnings
warnings.filterwarnings("ignore")
import good
import json
import os
import pandas as pd
import importlib
from good import s_runner
from good import helper
from concurrent.futures import ProcessPoolExecutor, as_completed
import multiprocessing as mp
from pathlib import Path

In [2]:
# =============================================================================
# Input settings
# =============================================================================

import os
import json
import importlib
import pandas as pd

# ---------------------------------------------------------
# Model region
# ---------------------------------------------------------
# This is the model region key, not one IPM node.
# For PJM, this maps to several GOOD nodes.
N_SCENARIO_WORKERS = 5
CPLEX_THREADS_PER_SCENARIO = 50
IPM_REGION = "MISO"
MODEL_REGION = IPM_REGION
STATE = IPM_REGION

YEAR_INPUT = 2030
MONTH_INPUT = 0
DAY_INPUT = 360

Discount_rate = 0.07
Lifetime = 25

# ---------------------------------------------------------
# Load EV data and scenarios
# ---------------------------------------------------------
# ---------------------------------------------------------
# Project paths
# ---------------------------------------------------------
def find_good_root():
    """
    Find the GOOD project root in a portable way.

    Priority:
    1. Use GOOD_ROOT environment variable if it exists.
    2. Search upward from the current working directory.
    """
    env_root = os.environ.get("GOOD_ROOT")
    if env_root:
        return Path(env_root).expanduser().resolve()

    current = Path.cwd().resolve()
    for folder in [current, *current.parents]:
        if (folder / "good").exists() and (folder / "Examples").exists():
            return folder

    raise FileNotFoundError(
        "Could not find GOOD project root. "
        "Set GOOD_ROOT to the path of your GOOD folder."
    )


GOOD_ROOT = find_good_root()
EV_DATA_DIR = GOOD_ROOT / "Examples" / "EVDATA"

ev_data_path = EV_DATA_DIR / f"{IPM_REGION}_EVDATA.json"

if not ev_data_path.exists():
    available_files = sorted(EV_DATA_DIR.glob("*.json"))
    available_names = [p.name for p in available_files]

    raise FileNotFoundError(
        f"Could not find EV data file:\n"
        f"  {ev_data_path}\n\n"
        f"Available files in {EV_DATA_DIR}:\n"
        f"  {available_names}"
    )

with ev_data_path.open("r") as f:
    ev_data = json.load(f)

SCENARIOS = helper.load_scenarios(IPM_REGION, YEAR_INPUT)

# ---------------------------------------------------------
# Load base graph and base policies
# ---------------------------------------------------------
BASE_GRAPH = good.graph.graph_from_json(f"Examples/Nodes/{IPM_REGION}_IPM.json")
BASE_POLICIES = good.utilities.read_json("Examples/policies.json")

# ---------------------------------------------------------
# Run settings
# ---------------------------------------------------------
TARGET_PEAK_GW = 150
ADOPTION_SCENARIOS = [
    # "slow",
    # "mid",
    "fast"
]
CHARGING_SCENARIOS = {
    # "midnight": {
    #     "profile": "timed_charging",
    #     "description": "Midnight timed charging",
    # },
    # "delay": {
    #     "profile": "max_delay",
    #     "description": "Maximum delay charging",
    # },
    "arrive": {
        "profile": "min_delay",
        "description": "Immediate arrival charging",
    },
    # "flex": {
    #     "profile": "load_leveling",
    #     "description": "Flexible load leveling",
    # },
}

# ---------------------------------------------------------
# GOOD regions used by this model region
# ---------------------------------------------------------
MISO_REGIONS = [
    "MIS_IL",
    "MIS_INKY",
    "MIS_IA",
    "MIS_MIDA",
    "MIS_LMI",
    "MIS_MO",
    "MIS_WUMS",
    "MIS_MNWI",
    "MIS_WOTA",
    "MIS_AMSO",
    "MIS_AR",
    "MIS_D_MS",
    "MIS_LA",
]

STATE_TO_REGIONS = {
    "MISO": MISO_REGIONS,
}

# ---------------------------------------------------------
# Battery storage distribution weights
# ---------------------------------------------------------
BATTERY_WEIGHTS = {
    "MIS_IL": 0.00,
    "MIS_INKY": 0.53,
    "MIS_IA": 0.10,
    "MIS_MIDA": 0.00,
    "MIS_LMI": 0.10,
    "MIS_MO": 0.00,
    "MIS_WUMS": 0.27,
    "MIS_MNWI": 0.00,
    "MIS_WOTA": 0.00,
    "MIS_AMSO": 0.00,
    "MIS_AR": 0.00,
    "MIS_D_MS": 0.00,
    "MIS_LA": 0.00,
}

assert abs(sum(BATTERY_WEIGHTS.values()) - 1.0) < 1e-6

# =============================================================================
# Policies used by run_one_scenario
# =============================================================================

# ---------------------------------------------------------
# Region-level retirement policies
# ---------------------------------------------------------
# These are for single-state regions.
# PJM state-specific retirement should come from the scenario JSON
# or from s_runner if you add that support there.
RETIREMENT_POLICIES = {
    "MISO": {
        "coal": 0.21,
        "oil": 0.00,
        "natural gas turbine": 0.02,
        "natural gas combined cycle": 0.00,
        "nuclear": 0.00,
    }
}

# ---------------------------------------------------------
# Asset constraint policies
# ---------------------------------------------------------

ASSET_CONSTRAINT_POLICIES = {
    "MISO": {
    # MISO 2030 central logic:
    # MISO still has substantial coal and nuclear generation.
    # However, MISO expects significant coal retirements, so coal should not be forced
    # to operate as baseload in every hour.
    # Nuclear remains highly must-run because it provides reliable low-carbon generation.
    # Combined-cycle gas provides reliability but remains flexible.
    # Simple-cycle gas turbines and oil units are primarily peaking resources.

    "nuclear": {
        "must_run_fraction": 0.90,
        "ramp_rate": 0.03,
    },

    "coal": {
        "must_run_fraction": 0.18,
        "ramp_rate": 0.05,
    },

    "natural gas combined cycle": {
        "must_run_fraction": 0.10,
        "ramp_rate": 0.20,
    },

    "natural gas turbine": {
        "must_run_fraction": 0.00,
        "ramp_rate": 0.75,
    },

    "oil": {
        "must_run_fraction": 0.02,
        "ramp_rate": 0.30,
        },
    },
}

FLEXIBILITY_POLICIES = {
    "default": {
        # -----------------------------
        # V1G settings
        # -----------------------------
        "v1g": {
            "base_shift_cost": 0.0,
            "fixed_om_per_kw_year": 0.0,
            "shift_window_hours": 24,
        },

        # -----------------------------
        # V2G settings
        # -----------------------------
        "v2g": {
            "window_hours": 24,
            "energy_duration_hours": 1,
            "roundtrip_efficiency": 0.985,
            "base_shift_cost": 1.3784e-9,
            "fixed_om_per_kw_year": 0.0,
        },

        # -----------------------------
        # Stationary battery settings
        # -----------------------------
        "battery": {
            "duration_hours": 4,
            "charge_efficiency": 0.93,
            "discharge_efficiency": 0.92,
            "fixed_om_per_kw_year": 3.75,
            "cycling_cost_per_mwh": 0.01,
            "initial_soc_fraction": 0.50,
            "total_power_mw": 1176,
        },
    },
    "MISO": {
        "battery": {
            "total_power_mw": 1176,
        },
    },
}

# ---------------------------------------------------------
# Economic policies
# ---------------------------------------------------------
ECONOMIC_POLICIES = {
    "default": {
        "discount_rate": 0.07,
        "lifetime_years": 25,
        "import_operating_cost": 1.75e-8,
        "apply_crf_to_renewables": True,
        "apply_crf_to_storage": True,
        "renewable_fuels_for_crf": {"solar", "wind"},
    },

    "MISO": {},
}

# ---------------------------------------------------------
# Transmission policies
# ---------------------------------------------------------
TRANSMISSION_POLICIES = {
    "default": {
        "apply_distance_enhancement": True,
        "default_operating_cost": 2.222222222222222e-09,
        "default_efficiency": 0.90,
        "overwrite_existing_operating_cost": False,
        "overwrite_existing_efficiency": False,
    },
    "MISO": {},
}

# ---------------------------------------------------------
# Policy inputs passed directly to run_one_scenario
# ---------------------------------------------------------
# Important:
# For PJM, state_rps_policies must be None.
# Each PJM scenario already has cfg["state_rps_policies"].
# This is how rps_minus10, rps_base, and rps_plus10 work.
POLICY_INPUTS = {
    "state_rps_policies": None,
}

In [3]:
"""
Professional Scenario Runner - Loops Through All Adoption × Charging Scenarios
Runs all combinations of adoption levels and charging patterns separately.
Each combination gets its own folder.
"""
# =============================================================================
# Main Loop - Process Each Adoption × Charging Scenario Combination
# =============================================================================

importlib.reload(s_runner)


total_combinations = len(ADOPTION_SCENARIOS) * len(CHARGING_SCENARIOS)
total_runs = len(SCENARIOS) * total_combinations
print(f"\n{'#'*80}")
print(f"# RUNNING ALL ADOPTION × CHARGING SCENARIOS")
print(f"# Adoption scenarios: {len(ADOPTION_SCENARIOS)} ({', '.join(ADOPTION_SCENARIOS)})")
print(f"# Charging scenarios: {len(CHARGING_SCENARIOS)} ({', '.join(CHARGING_SCENARIOS.keys())})")
print(f"# Total combinations: {total_combinations}")
print(f"# Total runs: {total_runs} (= {len(SCENARIOS)} scenarios × {total_combinations} combinations)")
print(f"{'#'*80}\n")
# Track overall progress
all_results_tracker = {}

for adoption_level in ADOPTION_SCENARIOS:

    print(f"\n{'█'*80}")
    print(f"█ ADOPTION LEVEL: {adoption_level.upper()}")
    print(f"{'█'*80}\n")

    for charging_name, charging_config in CHARGING_SCENARIOS.items():

        # Set up this combination
        RESULTS_DIR = f"Output/{IPM_REGION}/scenario_results_{YEAR_INPUT}_{adoption_level}_{IPM_REGION}_{charging_name}"
        CHARGING_PROFILE = charging_config["profile"]


        ctx = s_runner.ScenarioRunContext(
            scenarios=SCENARIOS,
            base_graph=BASE_GRAPH,
            base_policies=BASE_POLICIES,
            state_to_regions=STATE_TO_REGIONS,
            ev_data=ev_data,
            battery_weights=BATTERY_WEIGHTS,
            results_dir=RESULTS_DIR,
            discount_rate=Discount_rate,
            lifetime=Lifetime,
            retirement_policies=RETIREMENT_POLICIES,
            asset_constraint_policies=ASSET_CONSTRAINT_POLICIES,
            solver_threads=CPLEX_THREADS_PER_SCENARIO,
        )
        # Create unique key for tracking
        combo_key = f"{adoption_level}_{charging_name}"

        print(f"\n{'='*80}")
        print(f"COMBINATION: {adoption_level.upper()} adoption + {charging_config['description']}")
        print(f"Results Directory: {RESULTS_DIR}")
        print(f"{'='*80}\n")

        # Create results directory
        os.makedirs(RESULTS_DIR, exist_ok=True)

        # Run all scenarios for this combination
        all_results = []
        failed_scenarios = []

        tasks = []

        for scenario_id in sorted(SCENARIOS.keys()):
            tasks.append({
                "scenario_id": scenario_id,
                "ctx": ctx,
                "year": YEAR_INPUT,
                "adoption": adoption_level,
                "charging": CHARGING_PROFILE,
                "charging_name": charging_name,
                "charging_description": charging_config["description"],
                "month": MONTH_INPUT,
                "day_duration": DAY_INPUT,
                "model_region": IPM_REGION,
                "discount_rate": Discount_rate,
                "lifetime": Lifetime,
                "fix_peak": False,
                "target_peak_gw": TARGET_PEAK_GW,
                "peak_region_mode": "all",
                "peak_regions": None,
                "retirement_policy": None,
                "asset_constraint_policy": None,
                "use_state_default_retirement": True,
                "use_state_default_asset_constraints": True,
                "flexibility_policy": FLEXIBILITY_POLICIES,
                "use_state_default_flexibility": True,
                "economic_policy": ECONOMIC_POLICIES,
                "use_state_default_economic_policy": True,
                "transmission_policy": TRANSMISSION_POLICIES,
                "use_state_default_transmission_policy": True,
            })
       

        mp_context = mp.get_context("spawn")

        with ProcessPoolExecutor(
            max_workers=N_SCENARIO_WORKERS,
            mp_context=mp_context,
        ) as executor:

            futures = {
                executor.submit(s_runner.run_one_scenario_parallel_task, task): task["scenario_id"]
                for task in tasks
            }

            for future in as_completed(futures):
                scenario_id = futures[future]
                result = future.result()

                if result["ok"]:
                    all_results.append(result["row"])
                    print(f"  Scenario {scenario_id:>3} ✓")
                else:
                    print(f"  Scenario {scenario_id:>3} ✗ Error: {result['error']}")

                    failed_scenarios.append({
                        "scenario_id": scenario_id,
                        "error": result["error"],
                        "traceback": result["traceback"],
                    })
        
        
        # Save results for this combination
        all_results_df = pd.DataFrame(all_results)
        summary_path = os.path.join(RESULTS_DIR, "all_scenarios_summary.csv")
        all_results_df.to_csv(summary_path, index=False)

        # Save failed scenarios if any
        if failed_scenarios:
            failed_df = pd.DataFrame(failed_scenarios)
            failed_path = os.path.join(RESULTS_DIR, "failed_scenarios.csv")
            failed_df.to_csv(failed_path, index=False)

        # Store for later comparison
        all_results_tracker[combo_key] = all_results_df

        # Print summary for this combination
        print(f"\n  Summary:")
        print(f"    Successful: {len(all_results)}/{len(SCENARIOS)}")
        print(f"    Failed: {len(failed_scenarios)}/{len(SCENARIOS)}")
        print(f"    Saved to: {summary_path}")

        if failed_scenarios:
            print(f"    ⚠ Failed scenarios logged to: {failed_path}")


################################################################################
# RUNNING ALL ADOPTION × CHARGING SCENARIOS
# Adoption scenarios: 1 (fast)
# Charging scenarios: 1 (arrive)
# Total combinations: 1
# Total runs: 15 (= 15 scenarios × 1 combinations)
################################################################################


████████████████████████████████████████████████████████████████████████████████
█ ADOPTION LEVEL: FAST
████████████████████████████████████████████████████████████████████████████████


COMBINATION: FAST adoption + Immediate arrival charging
Results Directory: Output/MISO/scenario_results_2030_fast_MISO_arrive

EV state: MISO
Model region: MISO
STATE_TO_REGIONS key used: MISO
GOOD regions used: ['MIS_IL', 'MIS_INKY', 'MIS_IA', 'MIS_MIDA', 'MIS_LMI', 'MIS_MO', 'MIS_WUMS', 'MIS_MNWI', 'MIS_WOTA', 'MIS_AMSO', 'MIS_AR', 'MIS_D_MS', 'MIS_LA']
Base load peak scaling is OFF. Using original graph load.
Created RPS policies:
  rps_MISO_IL: ratio=0.3, r